In [5]:
import json
import pandas as pd
from typing import Dict, List

In [10]:
with open("raw_data/2026_league_teams.json", "r") as f:
    league_teams = json.load(f)



In [55]:
with open("raw_data/2026_squads.json", "r") as f:
    squads_dict = json.load(f)

with open("raw_data/world_rankings.json", "r") as f:
    world_rankings = json.load(f)

with open("raw_data/domestic_league_winners.json", "r") as f:
    domestic_league_winners = json.load(f)

with open("raw_data/domestic_top_scorers.json", "r") as f:
    domestic_top_scorers = json.load(f)

In [14]:
def count_league_players(league_teams: Dict[str, List[str]], squad: str) -> Dict[str, int]:
    counts = {f"{league}_players": 0 for league in league_teams.keys()}
    for league, teams in league_teams.items():
        for team in teams:
            counts[f"{league}_players"] += squad.count(team)
    return counts

In [28]:
import re

def get_name_and_rank_dict():
    # South Korea (37)
    mapped_names = {
        "fr yugoslavia": "yugoslavia",
        "republic of ireland": "ireland",
        "bosnia and herzegovina": "bosnia herzegovina",
        "ivory coast": "cote d ivoire",
        "trinidad and tobago": "trinidad tobago",
        "serbia and montenegro": "serbia"
    }
    res = {}
    for name_rank in world_rankings["2026"]:
        number = re.findall(r"\d+", name_rank)[0]
        name = re.sub(r"\(\d+\)", "", name_rank).strip()
        name = mapped_names.get(name, name)
        res[name] = int(number)
    return res


In [71]:
def count_top_3_club(teams: List[str]):
    count = 0
    for league, top_3 in domestic_league_winners["2026"].items():
        for club in top_3:
            occurrences = teams.count(club)
            count += occurrences
    return count

def count_top_scorers(squad):
    count = 0
    for league, scorers in domestic_top_scorers["2026"].items():
        for scorer in scorers:
            if scorer in squad:
                count += 1
    return count

In [72]:
def extract_made_knockouts(country, country_data, min_dir_path="../data/worldcup/min"):
    feature_year = "2026"
    query_year = int(feature_year) - 4
    with open(f"{min_dir_path}/{query_year}.txt", "r") as f:
        text = f.read()
    for feature_name in ["qualified_last_time", "made_r16_last_time", "made_quarters_last_time", "made_semis_last_time", "made_final_last_time"]:
        feature_value = made_knockout_round_last_time(text, country, phase=feature_name)
        country_data[feature_name] = feature_value
    return country_data

def made_knockout_round_last_time(text, country, phase="made_final_last_time"):
    phase_map = {
        "made_r16_last_time": ["Round of 16", "Quarter-final"],
        "made_quarters_last_time": ["Quarter-final", "Semi-finals"],
        "made_semis_last_time": ["Semi-final", "Final"]
    }
    if phase == "qualified_last_time":
        if country.lower() in text.lower():
            return 1
        if country.lower().replace("-", "") in "united states" and "usa" in text.lower():
            return 1
        return 0

    if phase in phase_map:
        processed = text.split(phase_map[phase][0])[1].split(phase_map[phase][1])[0].strip().lower()
        if country.lower() in processed:
            return 1
        if country.lower().replace("-", "") in "united states" and "usa" in processed:
            return 1
        return 0
    elif phase == "made_final_last_time":
        processed = text.split("Final")[1].strip().lower()
        if country.lower() in processed:
            return 1
        if country.lower().replace("-", "") in "united states" and "usa" in processed:
            return 1
        return 0
    else:
        raise ValueError("Invalid phase specified")

In [73]:
results_dict = {}
world_rankings_dict = get_name_and_rank_dict()
for country, squad in squads_dict.items():
    results_dict[country] = count_league_players(league_teams, squad)
    if country in ["Mexico", "USA", "Canada"]:
        results_dict[country]["is_home_nation"] = 1
    else:
        results_dict[country]["is_home_nation"] = 0

    results_dict[country]["ranking"] = world_rankings_dict[country]
    teams = re.findall(r"\((.*?)\)", squad)
    results_dict[country]["n_unique_clubs"] = len(set(teams))
    results_dict[country]["players_finishing_top_three_europe_domestic"] = count_top_3_club(teams)
    results_dict[country]["top_goalscorers_count"] = count_top_scorers(squad)
    results_dict[country] = extract_made_knockouts(country, results_dict[country])
    

In [75]:
with open("preprocessed_data/2026_featurised_squads.json", "w") as f:
    json.dump(results_dict, f, indent=4)